### Introduction to Audio and Music Processing (CH-EAM-B)
# 04 - Human Hearing and Sound Attributes

Prof. Dr. Jakob Abeßer (jakob.abesser@uni-bamberg.de)

Last update: 04.05.2026

## Learning Objectives

- Compute and visualize temporal envelopes using the Hilbert transform and the RMS values
- Convert RMS values to digital sound levels in decibels (dB)
- Detect pitch using autocorrelation

## Audio Material

- **temp_env_piano.wav** - piano note sample from https://freesound.org/people/nanliu_music/sounds/847227/
- **temp_env_trumpet.wav** - trumpet note sample from https://freesound.org/people/MTG/sounds/357589/
- **temp_env_violin.wav** - trumpet note sample from https://freesound.org/people/TheFlyFishingFilmmaker/sounds/641703/

In [ ]:
!pip install wget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import wget
import zipfile
import os

import librosa
import librosa.display
import soundfile as sf

from pathlib import Path
from IPython.display import Audio, display, Markdown

In [ ]:
fn_list = ['temp_env_piano.wav',
           'temp_env_trumpet.wav',
           'temp_env_violin.wav']

# download zip file (if it has been downloaded already)
if not os.path.isfile('CH-EAM-B-Seminar-04_sounds.zip'):
    print('Please wait a couple of seconds ...')
    wget.download('https://github.com/CHBamberg/CH-EAM-B-SS-2026/raw/refs/heads/main/data/CH-EAM-B-Seminar-04_sounds.zip', 
                      out='CH-EAM-B-Seminar-04_sounds.zip', bar=None)
    print('CH-EAM-B-Seminar-04_sounds.zip downloaded successfully ...')
else:
    print('Files already exist!')

# if at least one of the audio files does not exist -> unzip zip file
if any([not os.path.isfile(_) for _ in fn_list]):
    print("Let's unzip the file ... ")
    assert os.path.isfile('CH-EAM-B-Seminar-04_sounds.zip')
    with zipfile.ZipFile('CH-EAM-B-Seminar-04_sounds.zip', 'r') as f:
        f.extractall('.')
        
    assert all([os.path.isfile(_) for _ in fn_list])
    print("All done :)")
else:
    print('All audio files exist.')

## Helper Functions

In [ ]:
def show_audio(y, sr, label=None):
    """ Show playback button to listen to audio file
    Args:
        y (np.ndarray): Audio samples
        sr (float): Sample rate (in Hz)
        label (str): Optional label
    """
    if label is not None:
        display(Markdown(f"**{label}**"))
    display(Audio(y, rate=sr, normalize=False))
    
def plot_waveform(y, sr, start_s=0.0, end_s=None, title="Waveform"):
    """ Plot waveform of audio recording or a segment thereof
    Args:
        y (np.ndarray): Audio samples
        sr (float): Sample rate (in Hz)
        start_s (float): Segement start (in s)
        end_s (float): Segment end (in s), if None: end of the file is used
        title (str): optional figure title
    """
    start = int(start_s * sr)
    end = len(y) if end_s is None else int(end_s * sr)
    end = min(end, len(y))
    start = max(0, start)

    t = np.arange(start, end) / sr

    plt.figure(figsize=(12, 3))
    plt.plot(t, y[start:end], linewidth=1)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.title(title)
    plt.tight_layout()
    plt.show()
    
def rescale_to_same_max_amp(a, b):
    peak_a = np.max(a)
    peak_b = np.max(b)

    # Rescale
    a = a * (peak_b / peak_a)

    return a

def envelope_using_hilbert_transform(x, window_size = 200):

    from scipy.signal import hilbert
    
    # Compute the analytic signal using the Hilbert transform
    analytic_signal = hilbert(x)
    
    # obtain the envelope signal from its absolute value
    env = np.abs(analytic_signal)

    # Apply temporal smoothing (Low-pass filter effect)
    env_smooth = np.convolve(env, np.ones(window_size)/window_size, mode='same')
    
    # Re-scale to original signal
    env_smooth = rescale_to_same_max_amp(env_smooth, np.abs(x))
    
    return env_smooth

In [ ]:
# Let's listen to the audio files
for fn_wav in fn_list:
    x, fs = librosa.load(fn_wav)
    plot_waveform(x, fs, title=os.path.basename(fn_wav))
    show_audio(x, fs)

## 1) Temporal Envelope

In [ ]:
# TASK 1a: Describe the overall shape of the three waveforms. How do they differ?

### 1.1.) Temporal Envelope using Hilbert Transform

In [ ]:
# TASK 1b: - Use envelope_using_hilbert_transform() function to compute the temporal envelope 
#            for all three signals. 
#          - Plot the audio waveform (background) and the temporal envelope (foreground) on top
#            - Example code:
#                plt.plot(x, 'b-', alpha=0.2)
#                plt.plot(env, 'r-')
#          - Describe the envelope's shape     

### 1.2.) Temporal Envelope using Root Mean Square (RMS)

**EXPLAIN**

In [ ]:
# TASK 1c: 
#   - compute the root mean square values (RMS) over frames of the input signal (check https://librosa.org/doc/main/generated/librosa.feature.rms.html)
#   - think about the corresponding time position of each frame
#   - rescale the RMS values to the waveform (hint: use rescale_to_same_max_amp())
#   - plot the orignal waveform (background) and the RMS envelop (foreground)
#     - example: plt.plot(x, 'b-', alpha=0.2)
#                plt.plot(..., rms, 'r-')
#   - how do the envelopes from both methods (Hilbert, RMS) differ?

## 2) Dynamics

### 2.1) RMS to Digital Sound Level ($\text{dB}_\text{FS}$)

In a digital system (like your Python array), sound level is measured in Decibels relative to Full Scale ($\text{dB}_\text{FS}$). Since the maximum possible amplitude is $1.0$, $0 \, \text{dB}_\text{FS}$ is the loudest possible digital signal.

You can compute the sound level in dB (decibels) according to:
$$L_{\mathrm{dB}} = 20 \cdot \log_{10}\left(\frac{x_{\text{RMS}}}{x_\text{ref}}\right)$$

In [ ]:
# TASK 2a) What is value of x_ref?

In [ ]:
# TASK 2b) Plot again the RMS envelope using the sound level on the y axis

## 3) Pitch

We will implement a simple pitch detector and re-synthesize a sinosoid with the estimated pitch to compare with our reference signals.

### 3.1.) Pitch detection with auto-correlation

Autocorrelation is the most common time-domain method for pitch estimation. 

AC compares a signal with a "delayed" version of itself. If you shift a periodic signal by exactly one period, the two signals will align perfectly, creating a "peak" in correlation.

In [ ]:
def autocorrelation_pitch(y, sr):
    # 1. Compute autocorrelation for a range of "lags": a "lag" means a shift in time by a certain amount

    # compute the auto-correlation function
    corr = np.correlate(y, y, mode='full')
    corr = corr[len(corr)//2:] # Keep only the positive half
    
    # Define a search range in the lag-space:
    #   the lag is inverse proportional to frequency
    #   so we define a minimum frequency (e.g. 50 Hz) and a maximum frequency (e.g. 1000 Hz) 
    #   and compute the corresponding lag values (given the sample rate)
    min_lag = sr // 1000  # 1000 Hz limit
    max_lag = sr // 50    # 50 Hz limit

    # 2. Find the highest peak in the search range
    peak_lag = np.argmax(corr[min_lag:max_lag]) + min_lag
    
    # 3. Convert lag (samples) to frequency (Hz)
    f0 = sr / peak_lag
    return f0

In [ ]:
# TASK 3b) Compute the fundamental frequency using the ACF-based method implemented above for all 3 examples

The relationship between MIDI pitch $p$ and fundamental frequency $f_0$ is given by 


$$f_0 = 440 \cdot 2^{\frac{p-69}{12}}$$

In [ ]:
# TASK 3c) 
#  - Compute the MIDI pitch values corresponding to the estimated fundamental frequency values 
#    (hint: rearrange the equation above for p)
#  - Round to the nearest integer
#  - Cross-check with https://inspiredacoustics.com/en/MIDI_note_numbers_and_center_frequencies
#  - What are the musical notes (e.g. E5) played on the three instruments?

Done :)